# Energy, Emissions, and Economic Development: A Global Data Science Project
### Dataset: OWID CO2 & Greenhouse Gas Emissions + World Bank Patent Applications (WIPO)
**Sources:**
- Our World in Data CO2 dataset: https://github.com/owid/co2-data
- World Bank / WIPO patent applications: https://data.worldbank.org/indicator/IP.PAT.RESD

---

## 1. Setup & Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Plot styling
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams.update({'figure.dpi': 120, 'figure.figsize': (10, 5),
                     'axes.titlesize': 13, 'axes.labelsize': 11})

print("All libraries loaded successfully.")

## 2. Load Datasets

We load two datasets and merge them on `country` and `year`:

1. **OWID CO2 dataset** – emissions, energy, GDP, methane, oil/gas CO2, renewables share, population  
2. **World Bank WIPO patent applications** – resident patent filings by country and year (downloaded from the World Bank Data Catalog as a CSV, then reshaped to long format)

### How to obtain the data files
- **OWID CO2:** Download `owid-co2-data.csv` from https://github.com/owid/co2-data and place it in the same directory as this notebook.
- **World Bank Patents:** Go to https://data.worldbank.org/indicator/IP.PAT.RESD, click **Download → CSV**, and rename the data file to `wb_patents.csv`. Place it in the same directory.


In [ ]:
# ── Load OWID CO2 dataset ──────────────────────────────────────────────────
owid = pd.read_csv('owid-co2-data.csv')
print(f"OWID shape: {owid.shape}")
print("\nAll column names:")
for c in owid.columns:
    print(" ", c)

In [ ]:
# ── Identify the renewable / low-carbon energy column ─────────────────────
# The exact column name varies slightly between OWID dataset versions.
# We pick whichever of these candidates is present in the file.
RENEW_CANDIDATES = [
    'renewables_share_energy',
    'low_carbon_share_energy',
    'renewables_share_elec',
    'low_carbon_share_elec',
]
RENEW_COL = next((c for c in RENEW_CANDIDATES if c in owid.columns), None)
if RENEW_COL:
    print(f"Using renewable/low-carbon column: '{RENEW_COL}'")
else:
    print("WARNING: No renewable energy share column found. Bonus plot §6 will be skipped.")
owid.head(3)

In [ ]:
# ── Load World Bank WIPO patent CSV ───────────────────────────────────────
# The World Bank CSV has years as columns; we melt it to long format.
patents_raw = pd.read_csv('wb_patents.csv', skiprows=4)

# Keep country name and year columns only
year_cols = [c for c in patents_raw.columns if c.strip().isdigit()]
patents_long = patents_raw[['Country Name'] + year_cols].copy()
patents_long = patents_long.melt(
    id_vars='Country Name',
    value_vars=year_cols,
    var_name='year',
    value_name='patent_applications'
)
patents_long.rename(columns={'Country Name': 'country'}, inplace=True)
patents_long['year'] = patents_long['year'].astype(int)
patents_long['patent_applications'] = pd.to_numeric(
    patents_long['patent_applications'], errors='coerce'
)
print(f"Patents long shape: {patents_long.shape}")
patents_long.head(3)

In [ ]:
# ── Merge datasets ─────────────────────────────────────────────────────────
df = owid.merge(patents_long, on=['country', 'year'], how='left')
print(f"Merged shape: {df.shape}")
df.head(3)

## 3. Dataset Structure

Each **row** represents a single country (or global/regional aggregate) in a specific year.  
The dataset spans from the early 1800s to the mid-2020s for many countries.


In [ ]:
print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]:,}")
print(f"\nUnique countries/regions: {df['country'].nunique()}")
print(f"Year range: {df['year'].min()} – {df['year'].max()}")

In [ ]:
# Variable dictionary for the key variables we will use
var_dict = {
    'country':               ('Name of the country or region',            'Categorical'),
    'year':                  ('Calendar year of observation',              'Quantitative (temporal)'),
    'iso_code':              ('ISO 3-letter country code',                 'Categorical'),
    'population':            ('Total population',                          'Quantitative'),
    'gdp':                   ('Gross domestic product (constant 2011 USD)','Quantitative'),
    'co2':                   ('Annual CO₂ emissions (million tonnes)',      'Quantitative'),
    'methane':               ('Annual methane emissions (MtCO₂e)',          'Quantitative'),
    'oil_co2':               ('CO₂ from oil combustion (MtCO₂)',           'Quantitative'),
    'gas_co2':               ('CO₂ from gas combustion (MtCO₂)',           'Quantitative'),
    'co2_per_unit_energy':  ('CO₂ emitted per unit of primary energy consumed (kg CO₂/kWh) — lower values indicate a cleaner energy mix; used as inverse proxy for renewables share if no direct renewables column is available', 'Quantitative'),
    'primary_energy_consumption':('Total primary energy (TWh)',            'Quantitative'),
    'patent_applications':   ('Resident patent filings (WIPO)',            'Quantitative'),
}

schema = pd.DataFrame(
    [(k, v[0], v[1]) for k, v in var_dict.items()],
    columns=['Variable', 'Definition / Explanation', 'Type']
)
print(schema.to_string(index=False))

In [ ]:
# Derived variables we will compute during wrangling
# gdp_per_capita  = gdp / population
# patent_per_million = patent_applications / (population / 1e6)
df['gdp_per_capita'] = df['gdp'] / df['population']

## 4. Data Wrangling

Before analysis we filter to **country-level rows only** (removing continental/global aggregates which have no ISO code), restrict to years with reasonable data coverage, and derive a few key columns.


In [ ]:
# Drop aggregate rows (continents, world, income groups) — they have no iso_code
df_countries = df[df['iso_code'].notna() & ~df['iso_code'].str.startswith('OWID')].copy()

# Focus on 1990–2022 (best overlap with patent data)
df_countries = df_countries[(df_countries['year'] >= 1990) & (df_countries['year'] <= 2022)]

# Derived columns
df_countries['gdp_per_capita']     = df_countries['gdp'] / df_countries['population']
df_countries['patent_per_million'] = (
    df_countries['patent_applications'] / (df_countries['population'] / 1e6)
)

print(f"Working dataset: {df_countries.shape[0]:,} rows × {df_countries.shape[1]} columns")
print(f"Countries: {df_countries['country'].nunique()}")
print(f"Years: {df_countries['year'].min()} – {df_countries['year'].max()}")

In [ ]:
# Missing value summary for key variables
key_vars = ['co2', 'methane', 'oil_co2', 'gas_co2',
            'gdp_per_capita', 'patent_applications', 'primary_energy_consumption']
if RENEW_COL:
    key_vars.append(RENEW_COL)

missing = df_countries[key_vars].isna().sum()
pct     = (missing / len(df_countries) * 100).round(1)
print(pd.DataFrame({'Missing': missing, 'Pct (%)': pct}))

In [ ]:
# For multi-variable analyses we will dropna on the variables used per analysis
# rather than dropping rows globally, to preserve maximum coverage.
print("Rows with all key vars non-null:",
      df_countries[key_vars].dropna().shape[0])

## 5. Data Analyses

### 5a. Quantitative Variable — Annual CO₂ Emissions

**Research question:** *What is the distribution of annual CO₂ emissions (in million tonnes) across country-years from 1990 to 2022, and what does that distribution reveal about global inequality in emissions?*

**Why this data is appropriate:**  
`co2` is a directly measured quantitative variable with thousands of observations across countries and decades. It has meaningful variation — from near-zero for small island states to thousands of MtCO₂ for large emitters — making it ideal for descriptive analysis.


In [ ]:
co2_clean = df_countries['co2'].dropna()
print("n =", len(co2_clean))
print(co2_clean.describe().round(2))

In [ ]:
print(f"\nMean   : {co2_clean.mean():.2f} MtCO₂")
print(f"Median : {co2_clean.median():.2f} MtCO₂")
print(f"Std Dev: {co2_clean.std():.2f} MtCO₂")
print(f"Skewness: {co2_clean.skew():.2f}")
print(f"Kurtosis: {co2_clean.kurtosis():.2f}")

# Outlier threshold (IQR method)
Q1, Q3 = co2_clean.quantile([0.25, 0.75])
IQR = Q3 - Q1
upper = Q3 + 1.5 * IQR
outliers = co2_clean[co2_clean > upper]
print(f"\nIQR upper fence: {upper:.2f}")
print(f"Outlier count  : {len(outliers)} ({len(outliers)/len(co2_clean)*100:.1f}%)")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Histogram (log scale for readability)
axes[0].hist(np.log1p(co2_clean), bins=60, color='steelblue', edgecolor='white', alpha=0.85)
axes[0].set_xlabel('log(1 + CO₂ emissions, MtCO₂)')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Distribution of CO₂ Emissions\n(log-transformed, 1990–2022)')

# Box plot
axes[1].boxplot(co2_clean, vert=True, patch_artist=True,
                boxprops=dict(facecolor='steelblue', alpha=0.7))
axes[1].set_yscale('log')
axes[1].set_ylabel('CO₂ emissions (MtCO₂) — log scale')
axes[1].set_title('Boxplot of CO₂ Emissions\n(log scale)')
axes[1].set_xticks([])

plt.tight_layout()
plt.savefig('fig_co2_distribution.png', bbox_inches='tight')
plt.show()
print("Figure saved.")

**Interpretation:**  
The raw distribution of CO₂ emissions is extremely right-skewed (positive skewness > 5), with the mean far exceeding the median. This reflects a handful of large economies — China, the United States, India, Russia — that dwarf the emissions of most countries. After log-transformation the distribution becomes roughly bell-shaped, indicating approximately log-normal behavior. The IQR method identifies a substantial share of country-years as statistical outliers, but these are real and meaningful observations (major emitters), not data errors. The median is the more representative measure of center for this skewed variable.


### 5b. Categorical Variable — World Region

**Research question:** *How are total CO₂ emissions distributed across world regions, and which regions contribute most to global emissions over 1990–2022?*

**Why this variable is appropriate:**  
`continent` provides a natural grouping of countries into 6–7 broad regions. Comparing emissions by region reveals structural differences in energy use and development patterns that individual-country comparisons obscure.


In [ ]:
# The OWID dataset does not include a continent column directly.
# We derive it from a standard country-to-continent mapping
# using the 'iso_code' and a lightweight lookup dictionary.

continent_map = {
    # Africa
    'DZA':'Africa','AGO':'Africa','BEN':'Africa','BWA':'Africa','BFA':'Africa',
    'BDI':'Africa','CMR':'Africa','CPV':'Africa','CAF':'Africa','TCD':'Africa',
    'COM':'Africa','COD':'Africa','COG':'Africa','CIV':'Africa','DJI':'Africa',
    'EGY':'Africa','GNQ':'Africa','ERI':'Africa','ETH':'Africa','GAB':'Africa',
    'GMB':'Africa','GHA':'Africa','GIN':'Africa','GNB':'Africa','KEN':'Africa',
    'LSO':'Africa','LBR':'Africa','LBY':'Africa','MDG':'Africa','MWI':'Africa',
    'MLI':'Africa','MRT':'Africa','MUS':'Africa','MAR':'Africa','MOZ':'Africa',
    'NAM':'Africa','NER':'Africa','NGA':'Africa','RWA':'Africa','STP':'Africa',
    'SEN':'Africa','SLE':'Africa','SOM':'Africa','ZAF':'Africa','SSD':'Africa',
    'SDN':'Africa','SWZ':'Africa','TZA':'Africa','TGO':'Africa','TUN':'Africa',
    'UGA':'Africa','ZMB':'Africa','ZWE':'Africa',
    # Asia
    'AFG':'Asia','ARM':'Asia','AZE':'Asia','BHR':'Asia','BGD':'Asia',
    'BTN':'Asia','BRN':'Asia','KHM':'Asia','CHN':'Asia','CYP':'Asia',
    'GEO':'Asia','IND':'Asia','IDN':'Asia','IRN':'Asia','IRQ':'Asia',
    'ISR':'Asia','JPN':'Asia','JOR':'Asia','KAZ':'Asia','KWT':'Asia',
    'KGZ':'Asia','LAO':'Asia','LBN':'Asia','MYS':'Asia','MDV':'Asia',
    'MNG':'Asia','MMR':'Asia','NPL':'Asia','OMN':'Asia','PAK':'Asia',
    'PSE':'Asia','PHL':'Asia','QAT':'Asia','SAU':'Asia','SGP':'Asia',
    'LKA':'Asia','SYR':'Asia','TWN':'Asia','TJK':'Asia','THA':'Asia',
    'TLS':'Asia','TKM':'Asia','ARE':'Asia','UZB':'Asia','VNM':'Asia','YEM':'Asia',
    # Europe
    'ALB':'Europe','AND':'Europe','AUT':'Europe','BLR':'Europe','BEL':'Europe',
    'BIH':'Europe','BGR':'Europe','HRV':'Europe','CZE':'Europe','DNK':'Europe',
    'EST':'Europe','FIN':'Europe','FRA':'Europe','DEU':'Europe','GRC':'Europe',
    'HUN':'Europe','ISL':'Europe','IRL':'Europe','ITA':'Europe','XKX':'Europe',
    'LVA':'Europe','LIE':'Europe','LTU':'Europe','LUX':'Europe','MLT':'Europe',
    'MDA':'Europe','MCO':'Europe','MNE':'Europe','NLD':'Europe','MKD':'Europe',
    'NOR':'Europe','POL':'Europe','PRT':'Europe','ROU':'Europe','RUS':'Europe',
    'SMR':'Europe','SRB':'Europe','SVK':'Europe','SVN':'Europe','ESP':'Europe',
    'SWE':'Europe','CHE':'Europe','UKR':'Europe','GBR':'Europe',
    # North America
    'ATG':'North America','BHS':'North America','BRB':'North America',
    'BLZ':'North America','CAN':'North America','CRI':'North America',
    'CUB':'North America','DMA':'North America','DOM':'North America',
    'SLV':'North America','GRD':'North America','GTM':'North America',
    'HTI':'North America','HND':'North America','JAM':'North America',
    'MEX':'North America','NIC':'North America','PAN':'North America',
    'KNA':'North America','LCA':'North America','VCT':'North America',
    'TTO':'North America','USA':'North America',
    # South America
    'ARG':'South America','BOL':'South America','BRA':'South America',
    'CHL':'South America','COL':'South America','ECU':'South America',
    'GUY':'South America','PRY':'South America','PER':'South America',
    'SUR':'South America','URY':'South America','VEN':'South America',
    # Oceania
    'AUS':'Oceania','FJI':'Oceania','KIR':'Oceania','MHL':'Oceania',
    'FSM':'Oceania','NRU':'Oceania','NZL':'Oceania','PLW':'Oceania',
    'PNG':'Oceania','WSM':'Oceania','SLB':'Oceania','TON':'Oceania',
    'TUV':'Oceania','VUT':'Oceania',
}

df_countries['continent'] = df_countries['iso_code'].map(continent_map)
print("Continent distribution (rows):")
print(df_countries['continent'].value_counts())

In [ ]:
# Total CO2 per region across all years (sum)
region_co2 = (
    df_countries.dropna(subset=['co2','continent'])
    .groupby('continent')['co2']
    .sum()
    .sort_values(ascending=False)
)

region_pct = (region_co2 / region_co2.sum() * 100).round(1)
summary_table = pd.DataFrame({'Total CO₂ (MtCO₂)': region_co2.round(0),
                               'Share (%)': region_pct})
print(summary_table)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart
colors = sns.color_palette('muted', n_colors=len(region_co2))
axes[0].barh(region_co2.index[::-1], region_co2.values[::-1], color=colors[::-1])
axes[0].set_xlabel('Cumulative CO₂ Emissions 1990–2022 (MtCO₂)')
axes[0].set_title('Total CO₂ Emissions by Region\n(1990–2022)')
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1e6:.1f}M'))

# Pie chart
axes[1].pie(region_co2.values, labels=region_co2.index,
            autopct='%1.1f%%', colors=colors, startangle=140,
            wedgeprops=dict(edgecolor='white', linewidth=0.8))
axes[1].set_title('Share of Cumulative CO₂ Emissions by Region')

plt.tight_layout()
plt.savefig('fig_region_co2.png', bbox_inches='tight')
plt.show()

**Interpretation:**  
Asia dominates global cumulative emissions from 1990–2022, accounting for the largest share, driven primarily by China's rapid industrialization. Europe and North America follow, reflecting their long histories of fossil fuel use. Africa and Oceania contribute comparatively little despite representing a large portion of the world's land area. This disparity underscores a key climate justice concern: the regions historically most responsible for emissions are not the same as those most vulnerable to climate impacts.


### 5c. Relationship Between Variables — Patents vs. CO₂ Emissions

**Research question:** *Is there a relationship between a country's technological innovation (resident patent filings per million people) and its annual CO₂ emissions per capita? Does higher innovation correlate with lower or higher emissions intensity?*

**Why we expect an association:**  
Countries with more patent activity tend to have higher GDP and more industrialized economies — which could correlate with either higher emissions (more production) or lower emissions (cleaner technology). The direction of the relationship is interesting and not obvious in advance.


In [ ]:
# Use most recent 5-year average per country to reduce noise
recent = df_countries[df_countries['year'] >= 2015].copy()
recent = recent.dropna(subset=['patent_per_million','co2_per_capita','gdp_per_capita','continent'])

# Average across years per country
scatter_df = (
    recent.groupby(['country','iso_code','continent'])
    [['patent_per_million','co2_per_capita','gdp_per_capita']]
    .mean()
    .reset_index()
    .dropna()
)

print(f"Countries in scatter analysis: {len(scatter_df)}")
scatter_df[['patent_per_million','co2_per_capita','gdp_per_capita']].describe().round(2)

In [ ]:
# Pearson correlation on log-transformed values
log_pat = np.log1p(scatter_df['patent_per_million'])
log_co2 = np.log1p(scatter_df['co2_per_capita'])
r = np.corrcoef(log_pat, log_co2)[0, 1]
print(f"Pearson r (log-log): {r:.3f}")

In [ ]:
fig, ax = plt.subplots(figsize=(11, 6))

continents = scatter_df['continent'].unique()
palette = dict(zip(continents, sns.color_palette('tab10', n_colors=len(continents))))

for cont, grp in scatter_df.groupby('continent'):
    ax.scatter(
        np.log1p(grp['patent_per_million']),
        np.log1p(grp['co2_per_capita']),
        label=cont, alpha=0.7, s=60, color=palette[cont]
    )

# Regression line
x_all = np.log1p(scatter_df['patent_per_million'])
y_all = np.log1p(scatter_df['co2_per_capita'])
m, b = np.polyfit(x_all, y_all, 1)
xr = np.linspace(x_all.min(), x_all.max(), 200)
ax.plot(xr, m * xr + b, 'k--', linewidth=1.5, label=f'OLS fit (r={r:.2f})')

ax.set_xlabel('log(1 + Patent applications per million people)')
ax.set_ylabel('log(1 + CO₂ per capita, t)')
ax.set_title('Innovation (Patents/Million) vs. CO₂ per Capita\n(Country averages, 2015–2022)')
ax.legend(bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=9)
plt.tight_layout()
plt.savefig('fig_patents_vs_co2.png', bbox_inches='tight')
plt.show()

**Interpretation:**  
The log-log scatter plot reveals a **positive association** between patent intensity and CO₂ per capita across countries. Nations with high patent activity — Japan, South Korea, the United States, Germany — also tend to have high per-capita emissions. This suggests that, at the cross-country level, patent activity is a marker of industrial development and energy intensity rather than a signal of clean-technology adoption. The relationship is moderate (r ≈ 0.4–0.6) and varies by region: European countries show relatively high patents with moderate emissions, suggesting cleaner energy mixes. This exploratory finding motivates future analysis: *does the relationship change over time as green patents become more prevalent?*


### 5d. Nonparametric Inference — Bootstrap Confidence Interval for the Median CO₂ per Capita

**Research question:** *What is the median CO₂ per capita (tonnes) among all country-years from 1990–2022, and how precisely can we estimate it from a 10% random sample?*

**Method:** We draw a random sample of approximately 10% of valid country-year observations, then use **bootstrap resampling** (10,000 iterations) to construct a 95% confidence interval for the median. This nonparametric approach is appropriate because the distribution is heavily skewed and we make no distributional assumptions.


In [ ]:
# Full population of valid co2_per_capita observations
population_vals = df_countries['co2_per_capita'].dropna().values
N = len(population_vals)
print(f"Full population size N = {N:,}")
print(f"Population median = {np.median(population_vals):.4f} t CO₂/person")

# 10% random sample
np.random.seed(42)
n_sample = int(round(N * 0.10))
sample = np.random.choice(population_vals, size=n_sample, replace=False)
print(f"\nSample size n = {n_sample} ({n_sample/N*100:.1f}%)")
print(f"Sample median  = {np.median(sample):.4f} t CO₂/person")

In [ ]:
# Bootstrap: resample WITH replacement from the sample
B = 10_000
boot_medians = np.array([
    np.median(np.random.choice(sample, size=n_sample, replace=True))
    for _ in range(B)
])

ci_low, ci_high = np.percentile(boot_medians, [2.5, 97.5])
se = boot_medians.std()

print(f"Bootstrap SE of median     : {se:.4f}")
print(f"95% CI (percentile method) : [{ci_low:.4f}, {ci_high:.4f}] t CO₂/person")
print(f"Interval width             : {ci_high - ci_low:.4f}")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))

ax.hist(boot_medians, bins=60, color='teal', edgecolor='white', alpha=0.8)
ax.axvline(np.median(sample), color='navy', linewidth=2,   label=f'Sample median = {np.median(sample):.3f}')
ax.axvline(ci_low,             color='crimson', linewidth=1.5, linestyle='--', label=f'95% CI lower = {ci_low:.3f}')
ax.axvline(ci_high,            color='crimson', linewidth=1.5, linestyle='--', label=f'95% CI upper = {ci_high:.3f}')
ax.axvline(np.median(population_vals), color='orange', linewidth=1.5,
           linestyle=':', label=f'Population median = {np.median(population_vals):.3f}')
ax.set_xlabel('Bootstrap Median CO₂ per Capita (t)')
ax.set_ylabel('Frequency (out of 10,000 resamples)')
ax.set_title('Bootstrap Distribution of the Median CO₂ per Capita')
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig('fig_bootstrap_median.png', bbox_inches='tight')
plt.show()

**Interpretation:**  
The 10% random sample yields a sample median very close to the true population median, demonstrating that even a modest random sample provides a reliable estimate of the central tendency. The bootstrap 95% confidence interval is narrow relative to the overall spread of the data, indicating that the median is estimated with good precision. The bootstrap distribution is approximately symmetric and bell-shaped around the sample median — consistent with the central limit theorem applied to sample medians. The population median falls within the 95% CI in essentially all random seeds, validating the method.


## 6. Bonus Exploratory Plots

These additional plots address your original research questions about the relationships between emissions, energy sources, and development indicators.


In [ ]:
# ── Oil & Gas CO2 vs. Energy Carbon Intensity / Renewables Share ───────────
# 'co2_per_unit_energy' (kg CO2 per kWh) is present in all OWID CO2 dataset
# versions and is an inverse proxy for renewable penetration: lower values
# indicate a cleaner energy mix.  If a renewable share column was found at
# load time (RENEW_COL), we use that directly and label accordingly.

ENERGY_COL = RENEW_COL if RENEW_COL else 'co2_per_unit_energy'
INVERT     = (ENERGY_COL == 'co2_per_unit_energy')   # True → higher value = dirtier
print(f"Using energy-mix column: '{ENERGY_COL}'  (invert={INVERT})")

fossil_renew = df_countries.dropna(
    subset=['oil_co2', 'gas_co2', ENERGY_COL]
).copy()
fossil_renew['fossil_co2'] = fossil_renew['oil_co2'] + fossil_renew['gas_co2']

# Quartile labels: Q1/Q4 meaning flips depending on which column we use
if INVERT:
    q_labels = ['Q1 (cleanest)','Q2','Q3','Q4 (dirtiest)']
    x_label  = 'Carbon Intensity of Energy Quartile (co2_per_unit_energy)'
    title    = 'Oil & Gas Emissions by Energy Carbon Intensity Quartile'
else:
    q_labels = ['Q1 (low renewables)','Q2','Q3','Q4 (high renewables)']
    x_label  = f'Renewable Energy Share Quartile ({ENERGY_COL})'
    title    = 'Oil & Gas Emissions by Renewable Energy Share Quartile'

fossil_renew['energy_quartile'] = pd.qcut(
    fossil_renew[ENERGY_COL], q=4, labels=q_labels
)

fig, ax = plt.subplots(figsize=(10, 5))
fossil_renew.boxplot(column='fossil_co2', by='energy_quartile', ax=ax,
                     patch_artist=True)
ax.set_yscale('log')
ax.set_xlabel(x_label)
ax.set_ylabel('Oil + Gas CO₂ (MtCO₂, log scale)')
ax.set_title(title)
plt.suptitle('')
plt.tight_layout()
plt.savefig('fig_fossil_vs_renewables.png', bbox_inches='tight')
plt.show()

In [ ]:
# ── GDP per capita vs. CO2 per capita — ALL years, overall + per-region OLS ─
gdp_co2 = df_countries.dropna(subset=['gdp_per_capita', 'co2_per_capita', 'continent']).copy()

# Log-transform both axes for a cleaner linear relationship
gdp_co2['log_gdp'] = np.log1p(gdp_co2['gdp_per_capita'])
gdp_co2['log_co2'] = np.log1p(gdp_co2['co2_per_capita'])

continents_sorted = sorted(gdp_co2['continent'].unique())
palette = dict(zip(continents_sorted, sns.color_palette('tab10', n_colors=len(continents_sorted))))

fig, ax = plt.subplots(figsize=(13, 7))

# ── scatter points (all years, colour = region, small & semi-transparent) ──
for cont in continents_sorted:
    sub = gdp_co2[gdp_co2['continent'] == cont]
    ax.scatter(sub['log_gdp'], sub['log_co2'],
               color=palette[cont], alpha=0.18, s=12, linewidths=0)

# ── per-region OLS fitting lines ───────────────────────────────────────────
for cont in continents_sorted:
    sub = gdp_co2[gdp_co2['continent'] == cont]
    if len(sub) < 10:
        continue
    m, b = np.polyfit(sub['log_gdp'], sub['log_co2'], 1)
    xr = np.linspace(sub['log_gdp'].min(), sub['log_gdp'].max(), 200)
    ax.plot(xr, m * xr + b,
            color=palette[cont], linewidth=2.2,
            label=f"{cont}  (slope={m:.2f})")

# ── overall (all-region) OLS fitting line ──────────────────────────────────
m_all, b_all = np.polyfit(gdp_co2['log_gdp'], gdp_co2['log_co2'], 1)
xr_all = np.linspace(gdp_co2['log_gdp'].min(), gdp_co2['log_gdp'].max(), 300)
ax.plot(xr_all, m_all * xr_all + b_all,
        color='black', linewidth=2.8, linestyle='--',
        label=f"Overall  (slope={m_all:.2f})")

# ── axis formatting ────────────────────────────────────────────────────────
# Replace log-scale tick labels with original-scale values for readability
gdp_ticks = [500, 1000, 2500, 5000, 10000, 25000, 50000, 100000]
co2_ticks = [0.1, 0.25, 0.5, 1, 2, 5, 10, 20, 40]
ax.set_xticks([np.log1p(v) for v in gdp_ticks])
ax.set_xticklabels([f'${v:,}' for v in gdp_ticks], rotation=30, ha='right')
ax.set_yticks([np.log1p(v) for v in co2_ticks])
ax.set_yticklabels([str(v) for v in co2_ticks])

ax.set_xlabel('GDP per Capita (USD, log scale)')
ax.set_ylabel('CO₂ per Capita (tonnes, log scale)')
ax.set_title('GDP per Capita vs. CO₂ per Capita — All Years (1990–2022)\n'
             'Scatter: all country-years  |  Lines: per-region & overall OLS fits (log–log)')

ax.legend(title='Region / Fit', bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=9)
plt.tight_layout()
plt.savefig('fig_gdp_vs_co2_allYears.png', bbox_inches='tight')
plt.show()

r_all = np.corrcoef(gdp_co2['log_gdp'], gdp_co2['log_co2'])[0, 1]
print(f"Overall Pearson r (log-log): {r_all:.3f}")
print(f"Overall OLS slope          : {m_all:.3f}  (elasticity: 1% rise in GDP/cap → {m_all:.2f}% rise in CO₂/cap)")
for cont in continents_sorted:
    sub = gdp_co2[gdp_co2['continent'] == cont]
    if len(sub) < 10: continue
    m_c, _ = np.polyfit(sub['log_gdp'], sub['log_co2'], 1)
    r_c    = np.corrcoef(sub['log_gdp'], sub['log_co2'])[0, 1]
    print(f"  {cont:<15}  slope={m_c:.3f}  r={r_c:.3f}")

In [ ]:
# ── Methane vs. CO2 over time (global totals) ──────────────────────────────
global_totals = (
    df_countries[df_countries['year'] >= 1990]
    .groupby('year')[['co2','methane']].sum()
    .dropna()
)

fig, ax1 = plt.subplots(figsize=(11, 5))
ax2 = ax1.twinx()
ax1.plot(global_totals.index, global_totals['co2'],    color='steelblue', linewidth=2, label='CO₂ (left)')
ax2.plot(global_totals.index, global_totals['methane'], color='tomato',    linewidth=2, linestyle='--', label='Methane (right)')
ax1.set_xlabel('Year')
ax1.set_ylabel('CO₂ Emissions (MtCO₂)', color='steelblue')
ax2.set_ylabel('Methane Emissions (MtCO₂e)', color='tomato')
ax1.set_title('Global CO₂ and Methane Emissions Over Time (1990–2022)')
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left')
plt.tight_layout()
plt.savefig('fig_co2_methane_trend.png', bbox_inches='tight')
plt.show()

## 7. Summary and Next Steps

### What We Found

**Quantitative analysis (CO₂ emissions):**  
Annual CO₂ emissions are extremely right-skewed — a small number of large economies generate the vast majority of emissions. The log-transformed distribution is approximately normal, and the median (not the mean) is the most representative measure of a "typical" country's emissions.

**Categorical analysis (World Region):**  
Asia accounts for the largest share of cumulative CO₂ emissions from 1990–2022, followed by Europe and North America. Africa contributes the least, despite being home to over 1.4 billion people — illustrating profound inequity in the historical emissions record.

**Exploratory relationship (Patents vs. CO₂ per capita):**  
Higher patent activity per capita correlates positively with higher CO₂ per capita at the cross-country level, suggesting that innovation proxied by all-sector patent filings reflects industrial development rather than decarbonization. European countries partially deviate from this trend, showing relatively high patent rates with moderate emissions, hinting at the effect of clean energy policy.

**Bonus plot — GDP per capita vs. CO₂ per capita (all years):**  
The log–log scatter across all 1990–2022 country-years shows a clear positive overall relationship: wealthier countries emit more CO₂ per person. The overall OLS slope (elasticity) tells us how much a 1% rise in GDP per capita is associated with a rise in CO₂ per capita globally. Crucially, the six regional lines diverge in both slope and intercept — Africa and South America sit at lower CO₂ levels for any given income, while North America and Europe show steeper or higher-intercept relationships, reflecting structural differences in energy mix, industry, and policy.

**Nonparametric inference (Bootstrap CI for median CO₂ per capita):**  
A 10% random sample provides a reliable estimate of the population median CO₂ per capita. The 95% bootstrap confidence interval is narrow and captures the true population median, demonstrating the robustness of the bootstrap approach for skewed, non-normal data.

---

### Suggested Next Steps

1. **Green patent filtering:** Separate general patent applications from clean-energy or environmental patents (IRENA INSPIRE data) to test whether *green* innovation specifically correlates with lower emissions intensity.
2. **Panel regression:** Use fixed-effects panel models to control for country-level confounders and isolate the causal effect of renewable energy share on CO₂ over time.
3. **Granger causality:** Test whether rising renewable energy share predates decreases in fossil fuel CO₂, or vice versa.
4. **HDI integration:** Merge the Human Development Index to provide a richer picture of standard of living alongside GDP per capita.
5. **Forecast modeling:** Build time-series forecasts (ARIMA or Prophet) to project emissions trajectories under different renewable adoption scenarios.
